# Preentrenamiento GPT-50M con datos locales

Entrena con `train.txt`, evalúa exclusivamente con `validation.txt` y registra `val_loss` cada 10 actualizaciones.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/workspace/notebooks")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), f"No existe {PROJECT_ROOT / 'pyproject.toml'}"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e",
    f"{PROJECT_ROOT}[training]", "matplotlib"
])
print("Proyecto instalado desde:", PROJECT_ROOT)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import tiktoken
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from llm_mini_lab.models import GPTModel
from llm_mini_lab.training import (
    GPT_CONFIG_50M,
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text,
)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SEED = 123
MAX_LENGTH = 128
BATCH_SIZE = 2
MAX_TOKENS = 1_000_000
MAX_UPDATES = MAX_TOKENS // (BATCH_SIZE * MAX_LENGTH)
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
EVAL_INTERVAL = 10
VAL_BATCHES = 10

torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Actualizaciones:", MAX_UPDATES)
print("Tokens previstos:", MAX_UPDATES * BATCH_SIZE * MAX_LENGTH)

In [ ]:
class LocalTextDataset(Dataset):
    def __init__(self, file_path, max_length):
        file_path = Path(file_path)
        if not file_path.is_file():
            raise FileNotFoundError(f"No se encontró el archivo: {file_path}")

        text = file_path.read_text(encoding="utf-8")
        tokenizer = tiktoken.get_encoding("gpt2")
        self.tokens = tokenizer.encode(
            text, allowed_special={"<|endoftext|>"}
        )
        self.max_length = max_length

        if len(self.tokens) <= max_length:
            raise ValueError(f"{file_path} no contiene suficientes tokens")

    def __len__(self):
        return (len(self.tokens) - 1) // self.max_length

    def __getitem__(self, index):
        start = index * self.max_length
        end = start + self.max_length
        inputs = torch.tensor(self.tokens[start:end], dtype=torch.long)
        targets = torch.tensor(self.tokens[start + 1:end + 1], dtype=torch.long)
        return inputs, targets


DATA_DIR = PROJECT_ROOT / "data" / "smollm_local"
train_dataset = LocalTextDataset(DATA_DIR / "train.txt", MAX_LENGTH)
val_dataset = LocalTextDataset(DATA_DIR / "validation.txt", MAX_LENGTH)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=0, generator=loader_generator
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    num_workers=0
)

xb, yb = next(iter(train_loader))
assert xb.shape == yb.shape == (BATCH_SIZE, MAX_LENGTH)
assert torch.equal(yb[:, :-1], xb[:, 1:])

print("Directorio de datos:", DATA_DIR)
print("Bloques de entrenamiento:", len(train_dataset))
print("Bloques de validación:", len(val_dataset))
print("Tokens de entrenamiento:", len(train_dataset.tokens))
print("Tokens de validación:", len(val_dataset.tokens))

In [ ]:
cfg = {**GPT_CONFIG_50M, "context_length": MAX_LENGTH}
model = GPTModel(cfg)
model.out_head.weight = model.tok_emb.weight
model = model.to(device)

n_params = sum(parameter.numel() for parameter in model.parameters())
assert n_params <= 50_000_000, f"El modelo tiene {n_params:,} parámetros"
print(f"Parámetros entrenables: {n_params:,} ({n_params / 1e6:.2f}M)")

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

In [ ]:
def evaluate(model, data_loader, max_batches):
    was_training = model.training
    model.eval()
    total_loss = 0.0
    batches_evaluated = 0

    with torch.inference_mode():
        for batch_idx, (inputs, targets) in enumerate(data_loader):
            if batch_idx >= max_batches:
                break

            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
            total_loss += loss.item()
            batches_evaluated += 1

    model.train(was_training)

    if batches_evaluated == 0:
        raise RuntimeError("El dataloader de validación no produjo batches")

    return total_loss / batches_evaluated

In [ ]:
model.train()
train_losses = []
val_losses = []
val_steps = []
tokens_seen = 0
update = 0
epoch = 0

while update < MAX_UPDATES:
    epoch += 1
    print(f"Comenzando época {epoch}")

    for inputs, targets in train_loader:
        update += 1
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Pérdida no finita en el paso {update}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_losses.append(loss.item())
        tokens_seen += inputs.numel()

        if update % EVAL_INTERVAL == 0:
            mean_val_loss = evaluate(model, val_loader, VAL_BATCHES)
            val_steps.append(update)
            val_losses.append(mean_val_loss)
            print(
                f"Step {update:04d}/{MAX_UPDATES} | "
                f"train_loss {loss.item():.4f} | val_loss {mean_val_loss:.4f}"
            )

        if update >= MAX_UPDATES:
            break

assert len(train_losses) == MAX_UPDATES
print(f"Entrenamiento completado: {tokens_seen:,} tokens en {epoch} época(s)")

In [ ]:
train_steps = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 7))
plt.plot(train_steps, train_losses, label="Train loss", alpha=0.55)
plt.plot(
    val_steps, val_losses, marker="o", linewidth=2,
    label=f"Validation loss (cada {EVAL_INTERVAL} steps)"
)
plt.title("Train y validation loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.yscale("log")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
prompt = text_to_token_ids("Artificial intelligence", tokenizer).to(device)
generated_ids = generate_text_simple(
    model, prompt, max_new_tokens=20, context_size=MAX_LENGTH
)
print("Muestra:", token_ids_to_text(generated_ids.cpu(), tokenizer))

checkpoint_path = PROJECT_ROOT / "checkpoints" / "gpt-50m-local.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": cfg,
    "updates": update,
    "tokens_seen": tokens_seen,
    "train_losses": train_losses,
    "val_steps": val_steps,
    "val_losses": val_losses,
}, checkpoint_path)
print("Checkpoint guardado en:", checkpoint_path)